# Preview Extraction Mixtures
Listen to enrollment, mixture, clean target, and clean interferer side by side.
Also shows what Whisper transcribes on the mixture vs the ground truth.

In [1]:
import os, sys
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import pandas as pd
import IPython.display as ipd
import whisper
from src.preprocess import preprocess_audio, normalize_text, TARGET_SAMPLE_RATE

c:\Users\jibra\Documents\GitHub\TargetTTS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load recipes
df = pd.read_csv("data/mixture_recipes/librispeech_extraction_recipes.csv")
MIX_DIR = "data/synthetic_mixtures/librispeech_extraction"
print(f"Loaded {len(df)} recipes")

Loaded 12500 recipes


In [3]:
# Load Whisper model (change to "base" for better accuracy)
model = whisper.load_model("tiny")
print("Whisper loaded")

Whisper loaded


In [4]:
def preview_sample(idx):
    row = df.iloc[idx]
    mix_id = row["mix_id"]
    sir = row["sir_level_db"]
    overlap = row["overlap_ratio"]

    mix_path = os.path.join(MIX_DIR, f"{mix_id}.wav")
    target_path = os.path.join(MIX_DIR, f"{mix_id}_target.wav")
    interf_path = os.path.join(MIX_DIR, f"{mix_id}_interf.wav")
    enroll_path = row["enrollment_audio_path"]

    print(f"{'='*70}")
    print(f"Sample {idx}: {mix_id}  |  SIR={sir} dB  |  Overlap={overlap}")
    print(f"Target speaker: {row['target_speaker_id']}")
    print(f"{'='*70}")

    # Enrollment
    print("\n1. Enrollment clip (different utterance, same speaker):")
    w = preprocess_audio(enroll_path).squeeze(0).numpy()
    display(ipd.Audio(w, rate=TARGET_SAMPLE_RATE))

    # Clean target
    print("\n2. Clean target audio:")
    print(f"   Transcript: {row['target_transcript']}")
    w = preprocess_audio(target_path).squeeze(0).numpy()
    display(ipd.Audio(w, rate=TARGET_SAMPLE_RATE))

    # Clean interferer
    print("\n3. Clean interferer audio:")
    print(f"   Transcript: {row['noise_transcript']}")
    w = preprocess_audio(interf_path).squeeze(0).numpy()
    display(ipd.Audio(w, rate=TARGET_SAMPLE_RATE))

    # Mixture
    print("\n4. Mixture (what the model hears):")
    w_mix = preprocess_audio(mix_path).squeeze(0).numpy()
    display(ipd.Audio(w_mix, rate=TARGET_SAMPLE_RATE))

    # Whisper transcription of the mixture
    result = model.transcribe(w_mix, language="en")
    pred = normalize_text(result["text"])
    ref = normalize_text(row["target_transcript"])

    print(f"\n5. Whisper output on mixture:")
    print(f"   Predicted:  {pred}")
    print(f"   Reference:  {ref}")
    print()

In [5]:
# Preview first 3 samples (change indices to browse)
for i in range(3):
    preview_sample(i)

Sample 0: mix_librispeech_00000  |  SIR=5 dB  |  Overlap=0.2
Target speaker: 908

1. Enrollment clip (different utterance, same speaker):



2. Clean target audio:
   Transcript: i love thee with a love i seemed to lose with my lost saints i love thee with the breath smiles tears of all my life and if god choose i shall but love thee better after death



3. Clean interferer audio:
   Transcript: but now here is a subject of which you will wonder at first why turner drew it at all



4. Mixture (what the model hears):



5. Whisper output on mixture:
   Predicted:  i love thee with the love i seemed to lose with my lost saints it is a subject of which i love thee with the breath smile tears of all my life you are one and if god choose i shall but love thee better after death
   Reference:  i love thee with a love i seemed to lose with my lost saints i love thee with the breath smiles tears of all my life and if god choose i shall but love thee better after death

Sample 1: mix_librispeech_00001  |  SIR=0 dB  |  Overlap=0.5
Target speaker: 2961

1. Enrollment clip (different utterance, same speaker):


...


5. Whisper output on mixture:
   Predicted:  observe we want to know mister geard was how you the pursuit of wisdom searching out on our women ever came to commit such a thing
   Reference:  observe again what care the law took in the pursuit of wisdom searching out the deep things of the world and applying them to the use of man

Sample 2: mix_librispeech_00002  |  SIR=0 dB  |  Overlap=0.2
Target speaker: 61

1. Enrollment clip (different utterance, same speaker):



2. Clean target audio:
   Transcript: master monceux the sheriff of nottingham was mightily put about when told of the rioting



3. Clean interferer audio:
   Transcript: tied to a woman



4. Mixture (what the model hears):



5. Whisper output on mixture:
   Predicted:  master monso the sheriff of nodding time was mightalized about when told of the rioting
   Reference:  master monceux the sheriff of nottingham was mightily put about when told of the rioting



In [ ]:
# Preview a specific sample by index
preview_sample(42)